# Spotify Churn Prediction

**Purpose:** Predict whether a Spotify user will churn (cancel subscription) or remain active.

**Dataset summary (per-row = user):**
- `user_id` — unique identifier  
- `gender` — Male / Female / Other  
- `age` — integer (years)  
- `country` — user country  
- `subscription_type` — Free, Premium, Family, Student  
- `listening_time` — minutes/day  
- `songs_played_per_day` — count/day  
- `skip_rate` — percent (0-100)  
- `device_type` — Mobile, Desktop, Web  
- `ads_listened_per_week` — integer  
- `offline_listening` — minutes  
- `is_churned` — target (0 = Active, 1 = Churned)

**Objective:** Build models to predict `is_churned`, analyze engagement patterns, and recommend actions to reduce churn.

**Notebook flow (high level):**
1. Imports & load data (this notebook)  
2. EDA & cleaning  
3. Feature engineering & encoding  
4. Modeling & hyperparameter search  
5. Evaluation & business insights


## 📦 Step 1: Import Required Libraries

Before we start working with the Spotify dataset, we need to import the Python libraries that will help us with:

- **Data manipulation & analysis**:  
  `pandas`, `numpy`

- **Data visualization**:  
  `matplotlib`, `seaborn`

- **Machine learning & preprocessing**:  
  `scikit-learn` (for train/test split, encoding, scaling, and building models)

These libraries provide all the essential tools for:
* Cleaning and exploring the dataset,
* Building and evaluating machine learning models,
* Visualizing patterns and relationships in the data.

➡️ Next, we will write a code cell to import these libraries.

## 📂 Step 2: Load the Dataset

In this step we will:

1. **Read the CSV file** that contains the Spotify churn data.  
2. Store it in a **pandas DataFrame (`df`)** for easy analysis and manipulation.  
3. Quickly check the **shape** (number of rows and columns) to confirm that the data loaded correctly.

➡️ The next code cell will use `pandas.read_csv()` to import the dataset.
Make sure the CSV file (for example `spotify_2025.csv`) is in the notebook’s working directory or provide the full path to the file.



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split , RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Metrics
from sklearn.metrics import (accuracy_score,
                            classification_report,
                            confusion_matrix ,
                            precision_score,
                            recall_score,
                            f1_score,
                            roc_auc_score)


# Classification Models
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight

from scipy.stats import randint, uniform



In [ ]:
#Load the dataset
file_path = '/kaggle/input/spotify-dataset-for-churn-analysis/spotify_churn_dataset.csv'
df = pd.read_csv(file_path)

## 🔎 Step 3: Initial Data Exploration

After loading the dataset, we need to perform a quick check to understand its structure:

1. **Preview the data** using `head()` to see the first few rows.  
2. **Check column information** with `info()` to view data types and detect obvious issues (like unexpected object types).  
3. **Summarize numeric columns** using `describe()` to see key statistics such as mean, min, max, etc.  
4. **Check for missing values** with `isnull().sum()` to identify columns that may require cleaning or imputation.  

➡️ The next code cell will run these quick inspection commands.


In [ ]:
df.head(3)

In [ ]:
# Display basic info
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

## 📊 Insights from Data Summary 

Based on the descriptive statistics of the dataset:

### 1️⃣ General Overview
- The dataset contains **8,000 rows** for each feature.
- The **index/ID column** ranges from **1 to 8,000**, confirming no missing rows.

### 2️⃣ Age
- **Mean Age:** ~**37.7 years** with a **std of ~12.7**.
- **Range:** 16 to 59 years.
- **Insight:** The data covers a wide working-age population, slightly skewed towards younger adults.

### 3️⃣ Feature 3 (e.g., Income / Value)
- **Mean:** ~**154** with a **std of ~84**.
- **Range:** 10 to 299.
- **Insight:** Large spread indicates **high variability**; possible presence of both low and high earners/values.

### 4️⃣ Feature 4 (e.g., Score / Points)
- **Mean:** ~**50**, **std ~28.4**, range 1–99.
- **Insight:** Roughly centered at 50 with **balanced distribution**.

### 5️⃣ Feature 5 (e.g., Ratio / Rate)
- **Mean:** **0.30**, **std ~0.17**, range 0.0–0.6.
- **Insight:** Values are **bounded between 0 and 0.6**; distribution likely **right-skewed** (majority lower).

### 6️⃣ Feature 6
- **Mean:** **6.94**, but **std ~13.6**, **max 49**.
- **Insight:** Extreme variation—possible **outliers** or skewed distribution (many zeros, few very high values).

### 7️⃣ Binary Features (7 & 8)
- Feature 7:
  - **Mean:** ~0.75 → **~75% of entries are 1**.
- Feature 8:
  - **Mean:** ~0.26 → **~26% of entries are 1**.
- **Insight:** These are **binary categorical variables** with **imbalanced classes**.

---

### 🔍 Next Steps:
- **Check for outliers** in Feature 6 and Feature 3 using boxplots.
- **Visualize distributions** (histograms) to confirm skewness.
- **Consider scaling/normalization** for models sensitive to magnitude differences.
- **Address class imbalance** for binary features if used in classification.


## 🎨 Step 4: Data Visualization


- **Data Quality**  
  - No missing values were detected.  
  - Data types are appropriate for analysis (numeric & categorical).

- **Key Numerical Insights**  
  - **Age:** Users range from **16–59 years**, with a median of **38**, showing a fairly balanced adult user base.  
  - **Listening Time & Songs Played:**  
    - Average daily listening time is about **154 minutes** (~2.5 hours).  
    - Median songs played per day is **50**, but some users play up to **99**.  
  - **Skip Rate:** Mean skip rate is **30%**, indicating moderate track skipping behavior.  
  - **Ads Listened per Week:** Most users hear **0 ads**, but some hear up to **49**, suggesting a large share of Premium subscribers.
  - **Offline Listening:** Median is **1 minute**, but some users listen offline for much longer, showing diverse usage patterns.

- **Binary/Target Features**  
  - **Offline Listening & Churn:** Both show **class imbalance** (many users with 0 offline minutes and many remaining active).  
  - **is_churned:** Preliminary check reveals a larger active (0) group than churned (1), requiring attention when building models.

- **Next Steps**  
  - Perform deeper **visual EDA** to confirm these findings and identify outliers.  
  - Prepare **feature engineering** and **data preprocessing** (encoding categorical variables, scaling numeric features) before modeling.

➡️ These insights guide our next phase: **data cleaning and preprocessing** to prepare for churn prediction modeling.


In [ ]:
# LETS DISTRIBUTE THE FEATURE on basis of data distribution  FOR EASY VISULIZATION
cat_col = ["is_churned","offline_listening","device_type","subscription_type","country","gender"]
cat_num = ["ads_listened_per_week","age"]
num = df.drop(columns = cat_col+cat_num)

In [ ]:
# count plot for categorical columns
for col in cat_col:
    # lets understand the churn disyribution
    print(f"churn data distribution for  {col} ")
    print(df.groupby(col)["is_churned"].value_counts())

    # visulization
    fig,ax = plt.subplots(1,2,figsize = (12,6))
    sns.countplot(x=col,data = df , palette = "Set2",hue = "is_churned", ax = ax[0])
    ax[0].set_title(f"count plot for {col}")

    ax[1].pie(df[col].value_counts(),labels = df[col].value_counts().index,autopct= "%0.01f%%")
    ax[1].set_title(f"distribution of {col}")
    plt.show()

In [ ]:
for col in num:
    # 1️⃣ Display distribution counts (for discrete columns)
    print(f"\n📊 Churn data distribution for: {col}")
    print(df[col].value_counts().head(10))  # show top 10 unique values

    # 2️⃣ Create side-by-side plots
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # --- Boxplot ---
    sns.boxplot(y=col, data=df, palette="Set2", ax=ax[0])
    ax[0].set_title(f"Boxplot for {col}", fontsize=12)
    ax[0].set_ylabel(col)

    # --- Histogram + KDE ---
    sns.histplot(df[col], color="orange", edgecolor="black", kde=True, ax=ax[1], bins=30)
    ax[1].set_title(f"Distribution of {col}", fontsize=12)
    ax[1].set_xlabel(col)

    plt.tight_layout()
    plt.show()


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
sns.pairplot(
    df,
    vars=['age', 'listening_time', 'songs_played_per_day', 'skip_rate'],
    hue='is_churned',
    diag_kind='kde',
    palette='coolwarm'
)
plt.suptitle("Pairwise Relationships by Churn", y=1.02)
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.kdeplot(df[df['is_churned']==0]['skip_rate'], label='Active', shade=True)
sns.kdeplot(df[df['is_churned']==1]['skip_rate'], label='Churned', shade=True)
plt.title("Skip Rate Distribution: Active vs Churned")
plt.xlabel("Skip Rate")
plt.ylabel("Density")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(
    data=df,
    x='age',
    hue='is_churned',
    multiple='stack',
    palette='coolwarm',
    bins=20
)
plt.title("Age Distribution by Churn Status")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
df.sample()

## 🧩 Machine Learning Classification Pipeline

This notebook demonstrates a **complete end-to-end classification pipeline**:

1. **Data Preparation**  
   - Splits the dataset into **features (X)** and **target (y)**.  
   - Separates **categorical** and **numerical** columns.

2. **Preprocessing**  
   - **Categorical columns:** One-Hot Encoding (`OneHotEncoder`) with `handle_unknown='ignore'`.  
   - **Numeric columns:** Standard scaling (`StandardScaler`).

3. **Model Selection & Hyperparameter Tuning**  
   - Multiple classification models are compared:  
     - Logistic Regression  
     - K-Nearest Neighbors  
     - Decision Tree  
     - Random Forest  
     - Gradient Boosting  
     - AdaBoost  
     - Extra Trees  
     - XGBoost  
     - CatBoost  
   - For selected models, **`RandomizedSearchCV`** tunes hyperparameters using cross-validation and `accuracy` as the main metric.

4. **Evaluation Metrics**  
   - **Accuracy**, **Precision**, **Recall**, **F1-score**, and **ROC-AUC** are reported for each model.

5. **Summary Table**  
   - Results from all models are summarized in a single DataFrame sorted by Accuracy.


In [ ]:
# ================================================================
# 🔧 Libraries
# ================================================================
import warnings
warnings.filterwarnings("ignore")  # silence convergence warnings

import pandas as pd
from scipy.stats import uniform, randint

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, AdaBoostClassifier)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ✅ SMOTE for class imbalance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline   # ⚠️ important: imblearn's Pipeline

# ================================================================
# 🎯 Features & Target
# ================================================================
X = df.drop(columns=['is_churned', 'user_id'])   # user_id is just an identifier
y = df['is_churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ================================================================
# ⚙️ Preprocessing
# ================================================================
categorical_cols = ['gender','country','subscription_type',
                    'device_type','offline_listening']
numeric_cols = ['age','listening_time','songs_played_per_day',
                'skip_rate','ads_listened_per_week']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ])

# ================================================================
# 🤖 Models
# ================================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, solver='saga', random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42, allow_writing_files=False)
}

# ================================================================
# 🔍 Hyperparameter Distributions
# ================================================================
param_distributions = {
    "Logistic Regression": {
        "classifier__C": uniform(0.001, 100),
        "classifier__penalty": ['l2'],
        "classifier__solver": ['lbfgs','saga'],
        "classifier__max_iter": [3000]
    },
    "K-Nearest Neighbors": {
        "classifier__n_neighbors": randint(3, 50),
        "classifier__weights": ['uniform', 'distance']
    },
    "Decision Tree": {
        "classifier__max_depth": randint(2, 50),
        "classifier__min_samples_split": randint(2, 20),
        "classifier__min_samples_leaf": randint(1, 10)
    },
    "Random Forest": {
        "classifier__n_estimators": randint(100, 600),
        "classifier__max_depth": randint(3, 50),
        "classifier__min_samples_split": randint(2, 20),
        "classifier__min_samples_leaf": randint(1, 10)
    },
    "Extra Trees": {
        "classifier__n_estimators": randint(100, 600),
        "classifier__max_depth": randint(3, 50),
        "classifier__min_samples_split": randint(2, 20),
        "classifier__min_samples_leaf": randint(1, 10)
    },
    "Gradient Boosting": {
        "classifier__n_estimators": randint(100, 500),
        "classifier__learning_rate": uniform(0.001, 0.5),
        "classifier__max_depth": randint(2, 10)
    },
    "AdaBoost": {
        "classifier__n_estimators": randint(50, 500),
        "classifier__learning_rate": uniform(0.001, 1)
    },
    "XGBoost": {
        "classifier__n_estimators": randint(100, 500),
        "classifier__learning_rate": uniform(0.001, 0.5),
        "classifier__max_depth": randint(2, 15),
        "classifier__subsample": uniform(0.5, 0.5),
        "classifier__colsample_bytree": uniform(0.5, 0.5)
    },
    "CatBoost": {
        "classifier__depth": randint(4, 12),
        "classifier__learning_rate": uniform(0.001, 0.5),
        "classifier__iterations": randint(200, 800)
    }
}

# ================================================================
# 🚀 Deep Search & Evaluation with SMOTE
# ================================================================
results, tuned_models = {}, {}

for name, classifier in models.items():
    print(f"\n🔍 Deep tuning for {name} ...")

    # Use imbalanced-learn Pipeline so SMOTE is applied *inside* CV folds
    model = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', classifier)
    ])

    if name in param_distributions:
        random_search = RandomizedSearchCV(
            model,
            param_distributions=param_distributions[name],
            n_iter=50,          # deeper search
            cv=5,
            n_jobs=-1,
            scoring='f1_weighted',  # f1 is more informative for imbalanced data
            random_state=42,
            verbose=1
        )
        random_search.fit(X_train, y_train)
        tuned_model = random_search.best_estimator_
        print(f"✅ Best params for {name}: {random_search.best_params_}")
    else:
        model.fit(X_train, y_train)
        tuned_model = model

    tuned_models[name] = tuned_model
    y_pred = tuned_model.predict(X_test)

    results[name] = {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall":    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "F1-Score":  f1_score(y_test, y_pred, average='weighted', zero_division=0),
        "ROC-AUC":   roc_auc_score(y_test, tuned_model.predict_proba(X_test)[:,1])
    }

# ================================================================
# 🏆 Final Comparison
# ================================================================
results_df = pd.DataFrame(results).T.sort_values(by="F1-Score", ascending=False)
print("\n✅ Final Model Comparison after Deep Search + SMOTE:")
print(results_df)